# PPO 与 GRPO Loss：从零实现（面向 LLM 对齐）

本 Notebook **自包含、可独立运行**，仅依赖 `torch` 与 `numpy`，不导入项目中的任何其他脚本。

我们聚焦 RLHF / 大模型强化学习中最常用的两种策略优化损失：

1. **PPO（Proximal Policy Optimization，近端策略优化）**
   - 裁剪式重要性采样目标（Clipped Surrogate Objective）
   - GAE（Generalized Advantage Estimation）优势估计
   - 价值函数（Critic）损失 + 熵正则
2. **GRPO（Group Relative Policy Optimization，组相对策略优化）**
   - DeepSeekMath / DeepSeek-R1 提出，**去掉价值网络（Critic）**
   - 用「同一 prompt 采样一组回答」的组内相对奖励作为优势
   - 显式 per-token KL 惩罚（无偏 k3 估计）约束到参考模型

> 约定：在 LLM 场景下损失是 **token 级** 的。记号：`B`/`N` = 序列数，`T` = 生成 token 数，
> `mask` 标记「模型生成的回答 token」（prompt 与 padding 处为 0）。

## 参考
- Schulman et al., 2017, *Proximal Policy Optimization Algorithms*
- Schulman et al., 2016, *High-Dimensional Continuous Control Using GAE*
- Shao et al., 2024, *DeepSeekMath*（提出 GRPO）
- DeepSeek-AI, 2025, *DeepSeek-R1*

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)


def masked_mean(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """在 mask==1 的位置求均值（忽略 prompt / padding token）。"""
    return (values * mask).sum() / mask.sum().clamp(min=1.0)


print("torch:", torch.__version__)

## 1. PPO

### 1.1 核心思想

策略梯度朴素目标是 $\mathbb{E}[\log\pi_\theta(a|s)\,\hat A]$，一步更新过大会把策略带崩。
PPO 用**新旧策略概率比**

$$r_t(\theta)=\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}=\exp(\log\pi_\theta-\log\pi_{\theta_{old}})$$

并对其裁剪，得到 Clipped Surrogate Objective：

$$L^{CLIP}=\mathbb{E}_t\big[\min(r_t\hat A_t,\ \mathrm{clip}(r_t,1-\epsilon,1+\epsilon)\hat A_t)\big]$$

### 1.2 GAE 优势估计

$$\delta_t=r_t+\gamma V(s_{t+1})-V(s_t),\qquad \hat A_t=\sum_{l\ge0}(\gamma\lambda)^l\delta_{t+l}$$

In [ ]:
def compute_gae(rewards, values, mask, gamma=0.99, lam=0.95):
    """逐 token 计算 GAE 优势与 returns（LLM 中奖励通常加在序列末尾）。

    Args:
        rewards: [B, T] 每个 token 的奖励（多数为 0，末 token 为序列奖励）
        values:  [B, T+1] critic 状态价值，最后一列为 bootstrap
        mask:    [B, T] 回答 token 为 1，其余为 0
    Returns:
        advantages: [B, T];  returns: [B, T] (= advantages + values[:, :-1])
    """
    B, T = rewards.shape
    advantages = torch.zeros(B, T)
    last_adv = torch.zeros(B)
    for t in reversed(range(T)):
        delta = rewards[:, t] + gamma * values[:, t + 1] * mask[:, t] - values[:, t]
        last_adv = delta + gamma * lam * mask[:, t] * last_adv
        advantages[:, t] = last_adv
    returns = advantages + values[:, :-1]
    return advantages, returns

### 1.3 PPO 三部分损失

$$L = L^{policy} + c_1 L^{value} - c_2\,\mathcal H[\pi_\theta]$$

- **策略损失**：裁剪目标取负号（最小化）。
- **价值损失**：critic 拟合 returns，配合 value clipping 更稳。
- **熵奖励**：鼓励探索，避免策略过早坍缩。

In [ ]:
def ppo_policy_loss(log_probs, old_log_probs, advantages, mask, clip_eps=0.2):
    """裁剪式策略损失（token 级），输入均为 [B, T]。"""
    adv = advantages.detach()
    mean = masked_mean(adv, mask)
    var = masked_mean((adv - mean) ** 2, mask)
    adv = (adv - mean) / (var.sqrt() + 1e-8)          # 优势归一化降方差

    ratio = torch.exp(log_probs - old_log_probs)      # r_t(theta)
    unclipped = ratio * adv
    clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
    return -masked_mean(torch.min(unclipped, clipped), mask)


def ppo_value_loss(values, old_values, returns, mask, clip_eps=0.2):
    """带裁剪的价值损失（token 级）。"""
    v_clipped = old_values + torch.clamp(values - old_values, -clip_eps, clip_eps)
    loss_unclipped = (values - returns) ** 2
    loss_clipped = (v_clipped - returns) ** 2
    return 0.5 * masked_mean(torch.max(loss_unclipped, loss_clipped), mask)


def ppo_loss(log_probs, old_log_probs, advantages, returns,
             values, old_values, entropy, mask,
             clip_eps=0.2, vf_coef=0.5, ent_coef=0.01):
    """组合 PPO 总损失，并返回各分项用于监控。"""
    pg = ppo_policy_loss(log_probs, old_log_probs, advantages, mask, clip_eps)
    vf = ppo_value_loss(values, old_values, returns, mask, clip_eps)
    ent = masked_mean(entropy, mask)
    total = pg + vf_coef * vf - ent_coef * ent
    stats = {"policy_loss": pg.item(), "value_loss": vf.item(),
             "entropy": ent.item(), "total_loss": total.item()}
    return total, stats

### 1.4 一个最小的 PPO 训练步

把上面的损失接到一个极简「策略头 + 价值头」的玩具模型上，验证梯度能正常回传、损失可下降。
真实场景中只需把玩具模型换成你的 LLM（policy 输出 token 分布、value 头输出每个 token 的价值）。

In [ ]:
class TinyActorCritic(nn.Module):
    """玩具版 Actor-Critic：给定状态特征，输出动作分布 logits 与状态价值。"""

    def __init__(self, feat_dim=8, vocab=5, hidden=32):
        super().__init__()
        self.backbone = nn.Sequential(nn.Linear(feat_dim, hidden), nn.Tanh())
        self.actor = nn.Linear(hidden, vocab)     # 每个 token 位置的动作分布
        self.critic = nn.Linear(hidden, 1)        # 每个 token 位置的价值

    def forward(self, feats):
        h = self.backbone(feats)
        logits = self.actor(h)                    # [B, T, vocab]
        values = self.critic(h).squeeze(-1)       # [B, T]
        return logits, values


B, T, feat_dim, vocab = 4, 6, 8, 5
feats = torch.randn(B, T, feat_dim)
actions = torch.randint(0, vocab, (B, T))         # 旧策略采样得到的 token
mask = torch.ones(B, T)

model = TinyActorCritic(feat_dim, vocab)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

# ===== 1) rollout：用「旧策略」记录 log prob / value（detach，不回传） =====
with torch.no_grad():
    logits, values = model(feats)
    dist = torch.distributions.Categorical(logits=logits)
    old_log_probs = dist.log_prob(actions)
    old_values = values
    bootstrap = torch.zeros(B, 1)
    values_ext = torch.cat([values, bootstrap], dim=1)
    rewards = torch.zeros(B, T)
    rewards[:, -1] = torch.tensor([1.0, -0.5, 0.2, 0.8])   # 末 token 给奖励
    advantages, returns = compute_gae(rewards, values_ext, mask)

# ===== 2) 多个 epoch 复用同一批数据更新策略（PPO 的精髓） =====
for epoch in range(4):
    logits, values = model(feats)
    dist = torch.distributions.Categorical(logits=logits)
    log_probs = dist.log_prob(actions)
    entropy = dist.entropy()
    loss, stats = ppo_loss(log_probs, old_log_probs, advantages, returns,
                           values, old_values, entropy, mask)
    opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch}: " + ", ".join(f"{k}={v:.4f}" for k, v in stats.items()))

## 2. GRPO

### 2.1 动机：去掉 Critic

PPO 需要训练一个与策略同规模的价值网络（Critic）来估计优势，显存与算力开销很大。
**GRPO** 的洞察：对同一 prompt 采样**一组**回答，用组内奖励的相对高低直接当优势，**完全不需要 Critic**。

### 2.2 组相对优势

对一个 prompt 采样 $G$ 个回答，奖励 $\{r_1,\dots,r_G\}$，第 $i$ 个回答中**每个 token 共享**同一优势：

$$\hat A_i=\frac{r_i-\mathrm{mean}(\{r_1,\dots,r_G\})}{\mathrm{std}(\{r_1,\dots,r_G\})+\varepsilon}$$

### 2.3 目标函数（带 KL 惩罚）

$$L^{GRPO}=-\,\mathbb{E}\big[\min(r_t\hat A,\ \mathrm{clip}(r_t,1-\epsilon,1+\epsilon)\hat A)-\beta\,\mathbb{D}_{KL}[\pi_\theta\|\pi_{ref}]\big]$$

KL 用 DeepSeekMath 的**无偏、恒正 k3 估计**（逐 token）：

$$\mathbb{D}_{KL}\approx e^{\log\pi_{ref}-\log\pi_\theta}-(\log\pi_{ref}-\log\pi_\theta)-1$$

In [ ]:
def grpo_group_advantages(rewards, group_size, eps=1e-6):
    """组内标准化得到序列级优势。

    Args:
        rewards: [N] 标量奖励，N = num_prompts * group_size，按 prompt 连续分组
        group_size: 每个 prompt 的回答数 G
    Returns:
        advantages: [N] 每条序列一个优势（后续广播到该序列所有 token）
    """
    grouped = rewards.view(-1, group_size)              # [num_prompts, G]
    mean = grouped.mean(dim=1, keepdim=True)
    std = grouped.std(dim=1, keepdim=True)
    adv = (grouped - mean) / (std + eps)
    return adv.reshape(-1)                              # [N]


def grpo_loss(log_probs, old_log_probs, ref_log_probs, advantages, mask,
              clip_eps=0.2, kl_coef=0.04):
    """GRPO token 级损失（无 Critic）。

    Args:
        log_probs/old_log_probs/ref_log_probs: [N, T] 策略/旧策略/参考模型 token log prob
        advantages: [N] 序列级优势（来自 grpo_group_advantages）
        mask:       [N, T] 回答 token 掩码
    """
    adv = advantages.unsqueeze(1)                       # [N, 1] 广播到所有 token
    ratio = torch.exp(log_probs - old_log_probs)
    unclipped = ratio * adv
    clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
    policy_term = torch.min(unclipped, clipped)         # 越大越好

    log_ratio_ref = ref_log_probs - log_probs
    kl = torch.exp(log_ratio_ref) - log_ratio_ref - 1.0  # k3 估计，恒正无偏

    per_token = policy_term - kl_coef * kl
    loss = -masked_mean(per_token, mask)
    stats = {"loss": loss.item(),
             "policy_term": masked_mean(policy_term, mask).item(),
             "kl": masked_mean(kl, mask).item()}
    return loss, stats

### 2.4 一个最小的 GRPO 训练步

注意这里**只有策略网络（Actor），没有价值网络（Critic）**。
对每个 prompt 采样一组回答 → reward model 打分 → 组内标准化得到优势 → 更新策略。

In [ ]:
class TinyActor(nn.Module):
    """只含策略头的玩具模型（GRPO 不需要 Critic）。"""

    def __init__(self, feat_dim=8, vocab=5, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(feat_dim, hidden), nn.Tanh(),
                                 nn.Linear(hidden, vocab))

    def forward(self, feats):
        return self.net(feats)                          # [N, T, vocab]


G, num_prompts = 3, 2
N, T = num_prompts * G, 5
feats = torch.randn(N, T, feat_dim)
actions = torch.randint(0, vocab, (N, T))
mask = torch.ones(N, T)

actor = TinyActor(feat_dim, vocab)
ref_actor = TinyActor(feat_dim, vocab)                  # 参考模型（冻结）
ref_actor.load_state_dict(actor.state_dict())
for p in ref_actor.parameters():
    p.requires_grad_(False)
opt = torch.optim.Adam(actor.parameters(), lr=1e-2)

# rollout：旧策略 log prob + 参考模型 log prob（均 detach）
with torch.no_grad():
    old_log_probs = torch.distributions.Categorical(logits=actor(feats)).log_prob(actions)
    ref_log_probs = torch.distributions.Categorical(logits=ref_actor(feats)).log_prob(actions)
    # 每条序列由 reward model 打分（按 prompt 连续分组）
    rewards = torch.tensor([0.9, 0.1, 0.5,   2.0, 1.0, 1.5])
    advantages = grpo_group_advantages(rewards, group_size=G)

print("组相对优势:", [round(a, 3) for a in advantages.tolist()])
for epoch in range(4):
    log_probs = torch.distributions.Categorical(logits=actor(feats)).log_prob(actions)
    loss, stats = grpo_loss(log_probs, old_log_probs, ref_log_probs, advantages, mask)
    opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch}: " + ", ".join(f"{k}={v:.4f}" for k, v in stats.items()))

## 3. PPO vs GRPO 对比

| 维度 | PPO | GRPO |
| --- | --- | --- |
| 价值网络 Critic | 需要（与策略同规模） | **不需要** |
| 优势估计 | GAE（依赖 Critic + 逐步奖励） | 组内奖励标准化（序列级） |
| 采样方式 | 单条轨迹 | 同一 prompt 采样一组（G 个）回答 |
| KL 约束 | 常并入奖励或软约束 | 显式 per-token KL（k3 估计）项 |
| 显存 / 算力 | 较高 | 较低（省去 Critic） |
| 典型应用 | 通用 RLHF（InstructGPT 等） | 数学 / 推理对齐（DeepSeekMath、DeepSeek-R1） |

**要点**：
- 两者共享同一套裁剪式重要性采样目标，差别主要在**优势怎么来**与**是否需要 Critic**。
- GRPO 把基线（baseline）从「学习得到的价值函数」换成「同组其他回答的平均奖励」，实现简单、省资源。
- 本 Notebook 的两个损失都以**最小可运行**形式给出：把玩具模型换成你的 LLM、把合成奖励换成 reward model 打分即可接入真实训练循环。